# 📑 OPP-115 Privacy Policy Classification
---

> Evaluate and compare multiple Transformer-based architectures (`BERT`, `RoBERTa`, `LegalBERT`, and `DeBERTa`) using custom loss weightings to effectively classify multi-label privacy policy segments.

In [ ]:
!pip -q install transformers accelerate

from google.colab import files
up = files.upload()

import os, shutil
os.makedirs("src/classifier", exist_ok=True)
os.makedirs("configs", exist_ok=True)
os.makedirs("data", exist_ok=True)
for f in ["data.py", "split.py", "metrics.py"]:
    shutil.move(f, f"src/classifier/{f}")
shutil.move("split.json", "configs/split.json")
shutil.move("opp115_segments.parquet", "data/opp115_segments.parquet")
open("src/__init__.py", "w").close()
open("src/classifier/__init__.py", "w").close()

Saving opp115_segments.parquet to opp115_segments.parquet


In [ ]:
!pip install --upgrade torch torchvision torchaudio

['Requirement already satisfied: torch in /usr/local/lib/python3.13/dist-packages (2.11.0+cu130)',
 'Collecting torch',
 '  Downloading torch-2.14.1-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (37 kB)',
 'Requirement already satisfied: torchvision in /usr/local/lib/python3.13/dist-packages (0.26.0+cu130)',
 'Collecting torchvision',
 '  Downloading torchvision-0.29.1-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.7 kB)',
 'Requirement already satisfied: torchaudio in /usr/local/lib/python3.13/dist-packages (2.11.0+cu130)',
 'Requirement already satisfied: filelock in /usr/local/lib/python3.13/dist-packages (from torch) (4.0.7)',
 'Requirement already satisfied: typing-extensions>=4.10.0 in /usr/local/lib/python3.13/dist-packages (from torch) (4.16.0)',
 'Requirement already satisfied: setuptools>=77.0.3 in /usr/local/lib/python3.13/dist-packages (from torch) (80.10.2)',
 'Requirement already satisfied: sympy>=1.13.3 in /usr/local/lib/python3.13/dist-packages (from torch) (1.14.0)

In [ ]:
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)

from src.classifier.data import CATEGORIES, clean_text
from src.classifier.split import load_split, apply_split
from src.classifier.metrics import per_class_report

SEED = 42
set_seed(SEED)
MODEL = "bert-base-uncased"
MAXLEN = 256

df = pd.read_parquet("/content/data/opp115_segments.parquet")
df["text"] = df["text"].map(clean_text)
parts = apply_split(df, load_split())
print({k: len(v) for k, v in parts.items()})

tok = AutoTokenizer.from_pretrained(MODEL)


class DS(torch.utils.data.Dataset):
    def __init__(self, d):
        self.enc = tok(list(d["text"]), truncation=True, max_length=MAXLEN)
        self.y = d[CATEGORIES].values.astype("float32")

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.y[i]
        return item


train_ds, val_ds = DS(parts["train"]), DS(parts["val"])

{'train': 2550, 'val': 579, 'test': 663}


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def compute_metrics(p):
    probs = 1 / (1 + np.exp(-p.predictions))
    return {"macro_f1": f1_score(p.label_ids.astype(int), (probs > 0.5).astype(int),
                                 average="macro", zero_division=0)}


## 🚀 Experiment 1: Baseline BERT Model
---
*Testing standard multi-label classification using the `bert-base-uncased` architecture.*

In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(CATEGORIES), problem_type="multi_label_classification")

args = TrainingArguments(
    output_dir="out", num_train_epochs=5, learning_rate=3e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    weight_decay=0.01, warmup_steps=100, fp16=True, seed=SEED,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    save_total_limit=1, report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.262133,0.110061
2,No log,0.169459,0.431633
3,No log,0.139590,0.630465
4,0.242007,0.132078,0.627005
5,0.242007,0.127327,0.663115


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=800, training_loss=0.18549339532852172, metrics={'train_runtime': 265.5454, 'train_samples_per_second': 48.014, 'train_steps_per_second': 3.013, 'total_flos': 1372774667888568.0, 'train_loss': 0.18549339532852172, 'epoch': 5.0})

In [ ]:
out = trainer.predict(val_ds)
val_probs = 1 / (1 + np.exp(-out.predictions))
report = per_class_report(parts["val"][CATEGORIES].values, (val_probs > 0.5).astype(int), CATEGORIES)
report.to_csv("e4_bert_val_t05.csv")
np.save("val_probs_bert.npy", val_probs)
trainer.save_model("bert_model")
tok.save_pretrained("bert_model")
report

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,precision,recall,f1,support
First Party Collection/Use,0.849,0.893,0.871,177.0
Third Party Sharing/Collection,0.878,0.827,0.851,156.0
User Choice/Control,0.872,0.708,0.782,48.0
"User Access, Edit and Deletion",0.889,0.640,0.744,25.0
Data Retention,0.000,0.000,0.000,14.0
Data Security,1.000,0.667,0.800,27.0
Policy Change,1.000,0.762,0.865,21.0
Do Not Track,0.000,0.000,0.000,6.0
International and Specific Audiences,0.943,0.943,0.943,53.0
Other,0.848,0.713,0.775,157.0


## ⚖️ Experiment 2: BERT with Class-Weighting (v2)
---
*Addressing class imbalance using inverse frequency scaling to improve minority class detection.*

In [ ]:
ytr = parts["train"][CATEGORIES].values
pos = ytr.sum(axis=0)
neg = len(ytr) - pos
pos_weight = torch.tensor(np.clip(neg / pos, 1, 10), dtype=torch.float32)
print(dict(zip(CATEGORIES, pos_weight.numpy().round(1))))


class WeightedTrainer(Trainer):
    def __init__(self, *a, pos_weight=None, **kw):
        super().__init__(*a, **kw)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(CATEGORIES), problem_type="multi_label_classification")

args = TrainingArguments(
    output_dir="out2", num_train_epochs=12, learning_rate=3e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    weight_decay=0.01, warmup_steps=100, fp16=True, seed=SEED,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    save_total_limit=1, report_to="none",
)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorWithPadding(tok),
                          compute_metrics=compute_metrics, pos_weight=pos_weight)
trainer.train()

{'First Party Collection/Use': np.float32(2.1), 'Third Party Sharing/Collection': np.float32(3.0), 'User Choice/Control': np.float32(9.3), 'User Access, Edit and Deletion': np.float32(10.0), 'Data Retention': np.float32(10.0), 'Data Security': np.float32(10.0), 'Policy Change': np.float32(10.0), 'Do Not Track': np.float32(10.0), 'International and Specific Audiences': np.float32(10.0), 'Other': np.float32(2.7)}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.554257,0.533454
2,No log,0.361932,0.724859
3,No log,0.310254,0.788624
4,0.495927,0.350261,0.796716
5,0.495927,0.315736,0.810461
6,0.495927,0.287162,0.805286
7,0.139057,0.310636,0.816210
8,0.139057,0.309293,0.813333
9,0.139057,0.313039,0.821125
10,0.066849,0.313520,0.831033


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1920, training_loss=0.19205428957939147, metrics={'train_runtime': 1047.6376, 'train_samples_per_second': 29.209, 'train_steps_per_second': 1.833, 'total_flos': 3287914647721032.0, 'train_loss': 0.19205428957939147, 'epoch': 12.0})

In [ ]:
out = trainer.predict(val_ds)
val_probs = 1 / (1 + np.exp(-out.predictions))
yva = parts["val"][CATEGORIES].values
report_v2 = per_class_report(yva, (val_probs > 0.5).astype(int), CATEGORIES)
report_v2.to_csv("e4_bert_v2_val_t05.csv")
np.save("val_probs_bert_v2.npy", val_probs)
report_v2

,precision,recall,f1,support
First Party Collection/Use,0.793,0.932,0.857,177.0
Third Party Sharing/Collection,0.852,0.885,0.868,156.0
User Choice/Control,0.841,0.771,0.804,48.0
"User Access, Edit and Deletion",0.720,0.720,0.720,25.0
Data Retention,0.818,0.643,0.720,14.0
Data Security,0.926,0.926,0.926,27.0
Policy Change,0.900,0.857,0.878,21.0
Do Not Track,1.000,0.833,0.909,6.0
International and Specific Audiences,0.925,0.925,0.925,53.0
Other,0.784,0.739,0.761,157.0


## 📉 Experiment 3: Alternative Weighting Strategies
---
*Testing a softer, square-root based class-weight penalty to balance precision and recall.*

In [ ]:
# class weights: square root of (negatives / positives), capped at 5
ytr = parts["train"][CATEGORIES].values
pos = ytr.sum(axis=0)
neg = len(ytr) - pos
pos_weight = torch.tensor(np.clip(np.sqrt(neg / pos), 1, 5), dtype=torch.float32)
print(dict(zip(CATEGORIES, pos_weight.numpy().round(1))))

{'First Party Collection/Use': np.float32(1.4), 'Third Party Sharing/Collection': np.float32(1.7), 'User Choice/Control': np.float32(3.0), 'User Access, Edit and Deletion': np.float32(5.0), 'Data Retention': np.float32(5.0), 'Data Security': np.float32(4.1), 'Policy Change': np.float32(5.0), 'Do Not Track': np.float32(5.0), 'International and Specific Audiences': np.float32(3.6), 'Other': np.float32(1.6)}


In [ ]:

class WeightedTrainer(Trainer):
    def __init__(self, *a, pos_weight=None, **kw):
        super().__init__(*a, **kw)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(CATEGORIES), problem_type="multi_label_classification")

args = TrainingArguments(
    output_dir="out_b", num_train_epochs=12, learning_rate=3e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    weight_decay=0.01,warmup_steps=100, fp16=True, seed=SEED,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    save_total_limit=1, report_to="none",
)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorWithPadding(tok),
                          compute_metrics=compute_metrics, pos_weight=pos_weight)
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.390593,0.523703
2,No log,0.244993,0.759184
3,No log,0.200548,0.821010
4,0.350535,0.200309,0.828114
5,0.350535,0.182294,0.826409
6,0.350535,0.173962,0.839915
7,0.094245,0.177400,0.843300
8,0.094245,0.179903,0.840194
9,0.094245,0.181255,0.829713
10,0.047271,0.183742,0.831308


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1920, training_loss=0.13490692128737766, metrics={'train_runtime': 783.7199, 'train_samples_per_second': 39.045, 'train_steps_per_second': 2.45, 'total_flos': 3287914647721032.0, 'train_loss': 0.13490692128737766, 'epoch': 12.0})

In [ ]:
# score on validation at threshold 0.5
out = trainer.predict(val_ds)
val_probs_b = 1 / (1 + np.exp(-out.predictions))
yva = parts["val"][CATEGORIES].values
report_b = per_class_report(yva, (val_probs_b > 0.5).astype(int), CATEGORIES)
report_b.to_csv("e4_bert_sqrtw_val_t05.csv")
np.save("val_probs_bert_sqrtw.npy", val_probs_b)
report_b

,precision,recall,f1,support
First Party Collection/Use,0.857,0.915,0.885,177.0
Third Party Sharing/Collection,0.803,0.891,0.845,156.0
User Choice/Control,0.854,0.729,0.787,48.0
"User Access, Edit and Deletion",0.909,0.800,0.851,25.0
Data Retention,0.625,0.714,0.667,14.0
Data Security,0.889,0.889,0.889,27.0
Policy Change,0.857,0.857,0.857,21.0
Do Not Track,1.000,0.833,0.909,6.0
International and Specific Audiences,0.943,0.943,0.943,53.0
Other,0.839,0.764,0.800,157.0


### 📤 Upload Model to Hugging Face
---

In [ ]:
test_ds = DS(parts["test"])
out_t = trainer.predict(test_ds)
test_probs = 1 / (1 + np.exp(-out_t.predictions))
yte = parts["test"][CATEGORIES].values

pred_t = np.column_stack(
    [(test_probs[:, j] > thr[c]).astype(int) for j, c in enumerate(CATEGORIES)])
rep_test = per_class_report(yte, pred_t, CATEGORIES)
rep_test.to_csv("e4_bert_v2_test.csv")
np.save("test_probs_bert_v2.npy", test_probs)
rep_test

,precision,recall,f1,support
First Party Collection/Use,0.859,0.903,0.881,196.0
Third Party Sharing/Collection,0.789,0.882,0.833,144.0
User Choice/Control,0.600,0.774,0.676,62.0
"User Access, Edit and Deletion",0.870,0.800,0.833,25.0
Data Retention,0.714,0.500,0.588,10.0
Data Security,0.865,0.865,0.865,37.0
Policy Change,0.792,0.826,0.809,23.0
Do Not Track,1.000,0.667,0.800,6.0
International and Specific Audiences,0.815,0.869,0.841,61.0
Other,0.851,0.594,0.700,212.0


In [ ]:
from huggingface_hub import login
login()
#trainer.model.push_to_hub("dralsarrani/opp115-bert", private=True)
#tok.push_to_hub("dralsarrani/opp115-bert", private=True)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ivtpoib/model.safetensors:   0%|          | 14.2kB /  438MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/dralsarrani/opp115-bert/commit/fa1bfaeef878053ef02ade202230205c6f6612cf', commit_message='Upload tokenizer', commit_description='', oid='fa1bfaeef878053ef02ade202230205c6f6612cf', pr_url=None, repo_url=RepoUrl('https://huggingface.co/dralsarrani/opp115-bert', endpoint='https://huggingface.co', repo_type='model', repo_id='dralsarrani/opp115-bert'), pr_revision=None, pr_num=None)

## 🔄 Experiment 4: RoBERTa-based Classification
---

In [ ]:
MODEL = "roberta-base"          # second run: "nlpaueb/legal-bert-base-uncased"
TAG = "roberta"                 # second run: "legalbert"

# tokenizer and datasets must be rebuilt for the new model
tok = AutoTokenizer.from_pretrained(MODEL)
train_ds = DS(parts["train"])
val_ds = DS(parts["val"])

# class weights: same as BERT v2
ytr = parts["train"][CATEGORIES].values
pos = ytr.sum(axis=0)
neg = len(ytr) - pos
pos_weight = torch.tensor(np.clip(neg / pos, 1, 10), dtype=torch.float32)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, *a, pos_weight=None, **kw):
        super().__init__(*a, **kw)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(CATEGORIES), problem_type="multi_label_classification")

args = TrainingArguments(
    output_dir=f"out_{TAG}", num_train_epochs=12, learning_rate=3e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    weight_decay=0.01, warmup_steps=100, fp16=True, seed=SEED,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    save_total_limit=1, report_to="none",
)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorWithPadding(tok),
                          compute_metrics=compute_metrics, pos_weight=pos_weight)
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.439411,0.727609
2,No log,0.303575,0.754334
3,No log,0.282091,0.800487
4,0.430430,0.306205,0.824052
5,0.430430,0.283804,0.822430
6,0.430430,0.275042,0.827242
7,0.134819,0.266720,0.815274
8,0.134819,0.266412,0.839706
9,0.134819,0.305900,0.834696
10,0.070058,0.303270,0.839806


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1920, training_loss=0.17419335146745046, metrics={'train_runtime': 641.6652, 'train_samples_per_second': 47.688, 'train_steps_per_second': 2.992, 'total_flos': 3255594888802536.0, 'train_loss': 0.17419335146745046, 'epoch': 12.0})

In [ ]:
out = trainer.predict(val_ds)
val_probs = 1 / (1 + np.exp(-out.predictions))
yva = parts["val"][CATEGORIES].values
report = per_class_report(yva, (val_probs > 0.5).astype(int), CATEGORIES)
report.to_csv(f"e4_{TAG}_val_t05.csv")
np.save(f"val_probs_{TAG}.npy", val_probs)
report

,precision,recall,f1,support
First Party Collection/Use,0.838,0.904,0.870,177.0
Third Party Sharing/Collection,0.819,0.929,0.871,156.0
User Choice/Control,0.848,0.812,0.830,48.0
"User Access, Edit and Deletion",0.840,0.840,0.840,25.0
Data Retention,0.625,0.714,0.667,14.0
Data Security,0.852,0.852,0.852,27.0
Policy Change,0.900,0.857,0.878,21.0
Do Not Track,1.000,0.833,0.909,6.0
International and Specific Audiences,0.926,0.943,0.935,53.0
Other,0.761,0.790,0.775,157.0


## ⚖️ Experiment 5: Domain-Specific LegalBERT
---
*Evaluating a specialized pre-trained language model, `legal-bert-base-uncased`, designed for legal text structures.*

In [ ]:
MODEL = "nlpaueb/legal-bert-base-uncased"
TAG = "legalbert"

# tokenizer and datasets must be rebuilt for the new model
tok = AutoTokenizer.from_pretrained(MODEL)
train_ds = DS(parts["train"])
val_ds = DS(parts["val"])

# class weights: same as BERT v2
ytr = parts["train"][CATEGORIES].values
pos = ytr.sum(axis=0)
neg = len(ytr) - pos
pos_weight = torch.tensor(np.clip(neg / pos, 1, 10), dtype=torch.float32)

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, *a, pos_weight=None, **kw):
        super().__init__(*a, **kw)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(CATEGORIES), problem_type="multi_label_classification")

args = TrainingArguments(
    output_dir=f"out_{TAG}", num_train_epochs=12, learning_rate=3e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    weight_decay=0.01, warmup_steps=100, fp16=True, seed=SEED,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    save_total_limit=1, report_to="none",
)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorWithPadding(tok),
                          compute_metrics=compute_metrics, pos_weight=pos_weight)
trainer.train()

[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1920, training_loss=0.20332416693369548, metrics={'train_runtime': 762.2019, 'train_samples_per_second': 40.147, 'train_steps_per_second': 2.519, 'total_flos': 3373976644103352.0, 'train_loss': 0.20332416693369548, 'epoch': 12.0})

In [ ]:
out = trainer.predict(val_ds)
val_probs = 1 / (1 + np.exp(-out.predictions))
yva = parts["val"][CATEGORIES].values
report = per_class_report(yva, (val_probs > 0.5).astype(int), CATEGORIES)
report.to_csv(f"e4_{TAG}_val_t05.csv")
np.save(f"val_probs_{TAG}.npy", val_probs)
report

,precision,recall,f1,support
First Party Collection/Use,0.822,0.915,0.866,177.0
Third Party Sharing/Collection,0.834,0.936,0.882,156.0
User Choice/Control,0.683,0.854,0.759,48.0
"User Access, Edit and Deletion",0.833,0.800,0.816,25.0
Data Retention,0.833,0.714,0.769,14.0
Data Security,0.893,0.926,0.909,27.0
Policy Change,0.826,0.905,0.864,21.0
Do Not Track,1.000,1.000,1.000,6.0
International and Specific Audiences,0.942,0.925,0.933,53.0
Other,0.758,0.796,0.776,157.0


### 📤 Upload Model to Hugging Face
---

In [ ]:
import glob, json
ckpts = sorted(glob.glob("out_legalbert/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
print(ckpts)
state = json.load(open(ckpts[-1] + "/trainer_state.json"))
best = state["best_model_checkpoint"]
print(best, state["best_metric"])

['out_legalbert/checkpoint-1280']
out_legalbert/checkpoint-1280 0.8575760067704741


In [ ]:
from google.colab import userdata
token = userdata.get("hf_t")
REPO_ID = "dralsarrani/opp115-legalbert"   # use the same account as your BERT v2 repo

model = AutoModelForSequenceClassification.from_pretrained(best)
model.config.id2label = {i: c for i, c in enumerate(CATEGORIES)}
model.config.label2id = {c: i for i, c in enumerate(CATEGORIES)}
tok = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")

#model.push_to_hub(REPO_ID, token=token, private=False)
#tok.push_to_hub(REPO_ID, token=token)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...xc9x1t1/model.safetensors:   0%|          | 14.2kB /  438MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/dralsarrani/opp115-legalbert/commit/8cb474caf03bfdb8922a43c574697b4d062841df', commit_message='Upload tokenizer', commit_description='', oid='8cb474caf03bfdb8922a43c574697b4d062841df', pr_url=None, repo_url=RepoUrl('https://huggingface.co/dralsarrani/opp115-legalbert', endpoint='https://huggingface.co', repo_type='model', repo_id='dralsarrani/opp115-legalbert'), pr_revision=None, pr_num=None)

In [ ]:
from sklearn.metrics import f1_score

REPO_ID = "dralsarrani/opp115-legalbert"
tok = AutoTokenizer.from_pretrained(REPO_ID)      # DS uses this global tok
model = AutoModelForSequenceClassification.from_pretrained(REPO_ID)

test_ds = DS(parts["test"])
args = TrainingArguments(output_dir="out_test", per_device_eval_batch_size=64,
                         fp16=True, report_to="none")
trainer = Trainer(model=model, args=args, data_collator=DataCollatorWithPadding(tok))

out = trainer.predict(test_ds)
test_probs = 1 / (1 + np.exp(-out.predictions))
yte = parts["test"][CATEGORIES].values
pred = (test_probs > 0.5).astype(int)

report = per_class_report(yte, pred, CATEGORIES)
report.to_csv("e4_legalbert_test_t05.csv")
np.save("test_probs_legalbert.npy", test_probs)
print("micro-F1:", round(f1_score(yte, pred, average="micro", zero_division=0), 3))
report

config.json:   0%|          | 0.00/1.55k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/702k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

micro-F1: 0.801


,precision,recall,f1,support
First Party Collection/Use,0.846,0.923,0.883,196.0
Third Party Sharing/Collection,0.760,0.903,0.825,144.0
User Choice/Control,0.576,0.790,0.667,62.0
"User Access, Edit and Deletion",0.786,0.880,0.830,25.0
Data Retention,0.417,0.500,0.455,10.0
Data Security,0.903,0.757,0.824,37.0
Policy Change,0.792,0.826,0.809,23.0
Do Not Track,0.833,0.833,0.833,6.0
International and Specific Audiences,0.927,0.836,0.879,61.0
Other,0.765,0.708,0.735,212.0


## 🧪 Experiment 6: Robustness Verification (Seed 7)
---
*Validating the stability of the LegalBERT approach under a different initialization seed.*

In [ ]:
SEED = 7
MODEL = "nlpaueb/legal-bert-base-uncased"
TAG = "legalbert_s7"

# tokenizer and datasets must be rebuilt for the model
tok = AutoTokenizer.from_pretrained(MODEL)
train_ds = DS(parts["train"])
val_ds = DS(parts["val"])


In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, *a, pos_weight=None, **kw):
        super().__init__(*a, **kw)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(CATEGORIES), problem_type="multi_label_classification")

args = TrainingArguments(
    output_dir=f"out_{TAG}", num_train_epochs=12, learning_rate=3e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    weight_decay=0.01, warmup_steps=100, fp16=True, seed=SEED,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    save_total_limit=1, report_to="none",
)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorWithPadding(tok),
                          compute_metrics=compute_metrics, pos_weight=pos_weight)
trainer.train()

[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1920, training_loss=0.19477287928263345, metrics={'train_runtime': 540.3941, 'train_samples_per_second': 56.625, 'train_steps_per_second': 3.553, 'total_flos': 3376111491427920.0, 'train_loss': 0.19477287928263345, 'epoch': 12.0})

In [ ]:
out = trainer.predict(val_ds)
val_probs = 1 / (1 + np.exp(-out.predictions))
yva = parts["val"][CATEGORIES].values
report = per_class_report(yva, (val_probs > 0.5).astype(int), CATEGORIES)
report.to_csv(f"e4_{TAG}_val_t05.csv")
np.save(f"val_probs_{TAG}.npy", val_probs)
report

,precision,recall,f1,support
First Party Collection/Use,0.821,0.904,0.860,177.0
Third Party Sharing/Collection,0.845,0.942,0.891,156.0
User Choice/Control,0.774,0.854,0.812,48.0
"User Access, Edit and Deletion",0.870,0.800,0.833,25.0
Data Retention,1.000,0.571,0.727,14.0
Data Security,0.852,0.852,0.852,27.0
Policy Change,1.000,0.857,0.923,21.0
Do Not Track,1.000,0.833,0.909,6.0
International and Specific Audiences,0.927,0.962,0.944,53.0
Other,0.771,0.752,0.761,157.0


## 🚀 Experiment 7: DeBERTa-v3-small Classification
---

In [ ]:
!pip install -q sentencepiece tiktoken

In [ ]:
SEED = 42
MODEL = "microsoft/deberta-v3-small"
TAG = "deberta"

In [ ]:
# tokenizer and datasets must be rebuilt for the new model
tok = AutoTokenizer.from_pretrained(MODEL)
train_ds = DS(parts["train"])
val_ds = DS(parts["val"])

# class weights: same as BERT v2
ytr = parts["train"][CATEGORIES].values
pos = ytr.sum(axis=0)
neg = len(ytr) - pos
pos_weight = torch.tensor(np.clip(neg / pos, 1, 10), dtype=torch.float32)

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, *a, pos_weight=None, **kw):
        super().__init__(*a, **kw)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(CATEGORIES), problem_type="multi_label_classification")
model = model.float()

args = TrainingArguments(
    output_dir=f"out_{TAG}", num_train_epochs=12, learning_rate=3e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    weight_decay=0.01, warmup_steps=100, fp16=False, seed=SEED,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    save_total_limit=1, report_to="none",
)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorWithPadding(tok),
                          compute_metrics=compute_metrics, pos_weight=pos_weight)
trainer.train()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifie

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.584917,0.424971
2,No log,0.374681,0.699341
3,No log,0.313150,0.759325
4,0.518876,0.342564,0.780169
5,0.518876,0.330088,0.789909
6,0.518876,0.331604,0.793875
7,0.166933,0.368744,0.811266
8,0.166933,0.354359,0.814838
9,0.166933,0.370946,0.813737


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.584917,0.424971
2,No log,0.374681,0.699341
3,No log,0.313150,0.759325
4,0.518876,0.342564,0.780169
5,0.518876,0.330088,0.789909
6,0.518876,0.331604,0.793875
7,0.166933,0.368744,0.811266
8,0.166933,0.354359,0.814838
9,0.166933,0.370946,0.813737
10,0.083624,0.388498,0.809276


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1920, training_loss=0.21143045624097187, metrics={'train_runtime': 1591.1153, 'train_samples_per_second': 19.232, 'train_steps_per_second': 1.207, 'total_flos': 1622847264409632.0, 'train_loss': 0.21143045624097187, 'epoch': 12.0})

In [ ]:
out = trainer.predict(val_ds)
val_probs = 1 / (1 + np.exp(-out.predictions))
yva = parts["val"][CATEGORIES].values
report = per_class_report(yva, (val_probs > 0.5).astype(int), CATEGORIES)
report.to_csv(f"e4_{TAG}_val_t05.csv")
np.save(f"val_probs_{TAG}.npy", val_probs)
report

,precision,recall,f1,support
First Party Collection/Use,0.847,0.904,0.874,177.0
Third Party Sharing/Collection,0.821,0.910,0.863,156.0
User Choice/Control,0.783,0.750,0.766,48.0
"User Access, Edit and Deletion",0.731,0.760,0.745,25.0
Data Retention,0.643,0.643,0.643,14.0
Data Security,0.833,0.926,0.877,27.0
Policy Change,1.000,0.857,0.923,21.0
Do Not Track,1.000,0.833,0.909,6.0
International and Specific Audiences,0.923,0.906,0.914,53.0
Other,0.723,0.796,0.758,157.0


### 📤 Upload Model to Hugging Face
---

In [ ]:
import glob, json
ckpts = sorted(glob.glob("out_deberta/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
state = json.load(open(ckpts[-1] + "/trainer_state.json"))
best = state["best_model_checkpoint"]
print(best, state["best_metric"])

out_deberta/checkpoint-1760 0.8272673739755975


In [ ]:
from google.colab import userdata
token = userdata.get("hf_t")
REPO_ID = "dralsarrani/opp115-deberta"   # same account as your other repos

model = AutoModelForSequenceClassification.from_pretrained(best).float()
model.config.id2label = {i: c for i, c in enumerate(CATEGORIES)}
model.config.label2id = {c: i for i, c in enumerate(CATEGORIES)}
tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")

#model.push_to_hub(REPO_ID, token=token, private=False)
#tok.push_to_hub(REPO_ID, token=token)

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...rhgja7u/model.safetensors:   0%|          |  556kB /  568MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/dralsarrani/opp115-deberta/commit/7edef75723e27dfc08bb6093d043dd516194277b', commit_message='Upload tokenizer', commit_description='', oid='7edef75723e27dfc08bb6093d043dd516194277b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/dralsarrani/opp115-deberta', endpoint='https://huggingface.co', repo_type='model', repo_id='dralsarrani/opp115-deberta'), pr_revision=None, pr_num=None)